In [18]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDac-AGL/AgentDaC/notebooks


In [19]:
import socket

def find_free_port(host: str = "127.0.0.1") -> int:
    """Ask the OS for a free ephemeral port, then release it."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((host, 0))
        return s.getsockname()[1]

host = "127.0.0.1"
port = find_free_port(host)   # e.g. 51734
print(f"vLLM will use {host}:{port}")


vLLM will use 127.0.0.1:43893


In [20]:
import shlex

vllm_serve = [
    "vllm",
    "serve",
    "Qwen/Qwen3.5-9B",
    "--host",
    host,
    "--port",
    str(port),
    "--dtype",
    "bfloat16",
    "--max_model_len",
    "14336",
    "--max_num_seqs",
    "1024",
    "--enable_chunked_prefill",
    "--max_num_batched_tokens",
    "8192",
    "--enable_prefix_caching",
    "--logprobs_mode",
    "processed_logprobs",
    "--gpu_memory_utilization",
    "0.9",
    "--disable_log_stats",
    "--tensor_parallel_size",
    "1",
    "--seed",
    "0",
    "--override_generation_config",
    '{"temperature": 1.0, "top_k": 20, "top_p": 0.95, "repetition_penalty": 1.0, "max_new_tokens": 12288}',
    "--generation_config",
    "auto",
    "--additional_config",
    '{"gdn_prefill_backend": "triton"}',
]

def print_vllm_serve_command():
    print("vllm serve command:")
    print(shlex.join(vllm_serve))
    
print_vllm_serve_command()

vllm serve command:
vllm serve Qwen/Qwen3.5-9B --host 127.0.0.1 --port 43893 --dtype bfloat16 --max_model_len 14336 --max_num_seqs 1024 --enable_chunked_prefill --max_num_batched_tokens 8192 --enable_prefix_caching --logprobs_mode processed_logprobs --gpu_memory_utilization 0.9 --disable_log_stats --tensor_parallel_size 1 --seed 0 --override_generation_config '{"temperature": 1.0, "top_k": 20, "top_p": 0.95, "repetition_penalty": 1.0, "max_new_tokens": 12288}' --generation_config auto --additional_config '{"gdn_prefill_backend": "triton"}'


In [21]:
from src.inference import OAIClient

base_url = f"http://{host}:{port}/v1"

client = OAIClient(model_name="Qwen/Qwen3.5-9B", base_url=base_url)

# print(client.list_models())

In [22]:
from src.agents import PersistentAgent
from src.configs import PromptConfig, DecompConfig


prompt_config = PromptConfig(
    mode="path",
    system_root="../config_files/prompts/perst/v5_root.txt",
    system_inter="../config_files/prompts/perst/v5_root.txt",
    system_leaf="../config_files/prompts/perst/v5_leaf.txt",
    tasks_depleted=None,
)

decomp_config = DecompConfig(
    max_depth=1,
    max_tasks=4,
    max_rounds=5,
)

agent = PersistentAgent(
    client=client,
    prompt_config=prompt_config,
    decomp_config=decomp_config,
    additional_histories=True,
    force_thinking=True,
)

In [23]:
message = {"role": "user", "content": "What is the capital of France?"}

kwargs = {
    "temperature": 1.0,
    "top_p": 0.95,
    "extra_body": {
        "chat_template_kwargs": {"enable_thinking": False},
        "min_tokens": 5,
    },
}

response = await agent.chat(prompt=message, verbose=True, **kwargs)

Role:
SYSTEM
Content:
You are a careful reasoning assistant that solves the user's problem. You work in turns. On each turn you take exactly one action. One of the available actions is `think`: choose it whenever you want to reason before acting.

You are responsible for solving the main problem yourself. You can also use a sub-agent: a separate helper that you hand ONE piece of the work to. The sub-agent does that piece in its own context and sends back only its result. A sub-agent is a helper for parts of the problem - never a replacement that solves the whole thing for you.

# Why sub-agents exist

Sub-agents are NOT a way to avoid reasoning, and NOT a way to pass off the whole task. You keep working on the main problem yourself. Use a sub-agent when one part of the work is long, detailed, or self-contained, so that:

* that heavy part is done in the sub-agent's context instead of yours, keeping your own context clean, and
* your solution stays structured: you hand off a clear sub-t